In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-01-27_curate_controls_for_PheWAS_study"
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-13_get_conditions_of_cohorts"

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-01-27_curate_controls_for_PheWAS_study"
data2 ="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_statistic"

scratch="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2026-01-27_curate_controls_for_PheWAS_study"

#!mkdir ${scratch}

In [ ]:
#import and merge phecode mapping files

In [ ]:
def import_map_files():
   
    
    
    phe_icd= pd.read_csv(f"{data}/expanded_phecode.csv")
    
    phe = pd.read_csv(f"{data}/phecode_info.csv")
    
    phex = pd.read_csv(f"{data}/phecodex_info.csv")
    
    phex = phex.rename({"phecodex": "phecode"}, axis="columns")
    
    phex_icd = pd.read_csv(f"{data}/updated_phecodex_map.csv")
    
    
    return phe, phe_icd, phex, phex_icd
    

In [ ]:
def merge_phecodes_with_phenotype(phe, phe_icd):
    
    final = pd.merge(phe, phe_icd, how = "inner")
    
    return final

In [ ]:
#import cohort dataframe and wrangle into cohorts

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
def get_cond_cohort_df_dict(all_cond_df): 
    print("Grouping DataFrame into a dictionary...")

    # Define the columns you want to use for your composite key
    key_columns = ['condition_concept_id', 'standard_concept_name']

    # Use a dictionary comprehension with groupby to create the dictionary
    # - The 'key' will be a tuple: (condition_concept_id, standard_concept_name)
    # - The 'group_df' will be the DataFrame containing all rows for that key
    concept_groups_dict = {
        key: group_df 
        for key, group_df in all_cond_df.groupby(key_columns)
    }

    print(f"Successfully created a dictionary with {len(concept_groups_dict)} unique (ID, Name) keys.")
    
    return concept_groups_dict

In [ ]:
def filter_cond_df_dict(cohort_dict):
    
    
    master_df_of_final_cohorts = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")

    master_key_cols = ['condition_concept_id', 'standard_concept_name']

    # Create a set of tuples (key1, key2) from the master DataFrame.
    # This set will act as our "allow list".
    valid_keys_set = set(
        master_df_of_final_cohorts[master_key_cols].itertuples(index=False, name=None)
    )
    
    # 'concept_groups_dict' is the dictionary you created in the previous step
    # 'valid_keys_set' is the set we just created

    filtered_concept_dict = {
        key: group_df 
        for key, group_df in cohort_dict.items() 
        if key in valid_keys_set
    }

    print(f"Original dictionary had {len(cohort_dict)} items.")
    print(f"Filtered dictionary now has {len(filtered_concept_dict)} items.")

    # You can now work with your new, smaller dictionary
    # print(list(filtered_concept_dict.keys())[:5])
    
    return filtered_concept_dict

In [ ]:
#count the proportion of unqiue samples coded with an ICD9/10 code atleast 1 in each of my cohorts

In [ ]:
def get_icd_vocab_count_per_cohort(filtered_cohort_dict):
    
    icd_vocabs = ["ICD9CM", "ICD10CM"]

    results = []

    for key, df in filtered_cohort_dict.items():
        total_individuals = df["person_id"].nunique()
        mask_icd = df["source_vocabulary"].isin(icd_vocabs)
        individuals_with_icd = df.loc[mask_icd, "person_id"].nunique()
        prop_with_icd = individuals_with_icd / total_individuals if total_individuals > 0 else float("nan")

        results.append({
            "key": key,
            "total_individuals": total_individuals,
            "with_icd": individuals_with_icd,
            "prop_with_icd": prop_with_icd,
        })

    summary = pd.DataFrame(results)
    #summary.to_csv("cohort_source_vocab_count.csv")

    return summary


In [ ]:
#create a cohort concept_name/id to IDC9/10 code mapping file from filtered cohohort dictionary

In [ ]:
def get_cohort_concept_id_to_ICD_mapping(raw_cohort_df):   
    
    master_df_of_final_cohorts = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")
    
    # columns we want in the final df
    cols = ['condition_concept_id', 'standard_concept_name', "source_concept_code", "source_vocabulary"]

    # keep only ICD9/10 rows, then select the 3 columns and drop duplicates
    icd_df = (
        raw_cohort_df[raw_cohort_df["source_vocabulary"].isin(["ICD9CM", "ICD10CM"])]
        [cols]
        .dropna(subset=["source_concept_code"])   
        .drop_duplicates()
    )
    
    
    master_cohort_list = master_df_of_final_cohorts["condition_concept_id"]
    final = icd_df[icd_df["condition_concept_id"].isin(master_cohort_list)]
    
    
    return final


In [ ]:
#get percent of ICD overlap of my cohorts to phecodes dataframe

In [ ]:
# 1. get Unique condition–ICD and phecode–ICD

def get_unique_phecodes_and_cohort_ICDs(cohort_ICD_map,phe_map, phex_map):

    cond_codes = cohort_ICD_map[
        ["condition_concept_id", "standard_concept_name", "source_concept_code", "source_vocabulary"]]


    # combine phecode + phecodeX
    phe_all = pd.concat([phe_map, phex_map], ignore_index=True)

    phe_codes = phe_all[
        ["phecode", "description", "group", "concept_code", "vocabulary_id"]
    ].drop_duplicates()
    
    
    return cond_codes, phe_codes


In [ ]:
 def get_cohort_ICD_codes_count_per_unique_condition(cond_codes): 
        
        cohort_icd_count = (
            cond_codes.groupby(["condition_concept_id", "standard_concept_name"])["source_concept_code"].nunique().rename("n_my_icds").reset_index()) 
        
        return cohort_icd_count 
    
    
    

In [ ]:
# 4. get overlap count of cohort ICDs to phecode/phecodeX ICDS 

#join on ICD code + source/vocabulary to get overlap file

def get_cohort_phecode_ICD_overlap(cond_codes, phe_codes):
    overlap = cond_codes.merge(
        phe_codes,
        left_on=["source_concept_code", "source_vocabulary"],
        right_on=["concept_code", "vocabulary_id"],
        how="inner",
    )

    return overlap


# Count overlapping ICD codes for each (condition, phecode)/ count overlap file 
#group by unqiue ("condition_concept_id", "standard_concept_name", "phecode") group/ per each unique ICD code
#get a column with a list of the overlapping icds

def get_ICD_overlap_count(overlap_df):
    
    overlap_counts = (
        overlap_df
        .groupby(["condition_concept_id", "standard_concept_name", "phecode"])
        .agg(
            n_overlap=("source_concept_code", "nunique"),
            overlap_icds=("source_concept_code", lambda x: sorted(set(x)))  # list of ICDs
        )
        .reset_index()
    )
    return overlap_counts



#merge num of overlap ICD counts df with unique cohort ICD count file
# Add denominator and compute fraction

def get_cohort_phecode_ICD_overlap_frac_count(overlap_counts, cohort_icd_count):
    overlap_counts2 = overlap_counts.merge(
        cohort_icd_count,
        on= ["condition_concept_id", 'standard_concept_name'],
        how="inner"
    )

    #add frac overlap column
    overlap_counts2["frac_overlap"] = overlap_counts2["n_overlap"] / overlap_counts2["n_my_icds"]

    final = overlap_counts2[[ 
        "condition_concept_id", 
          'standard_concept_name',
          'phecode', 
          'n_my_icds',
          'n_overlap', 
          'frac_overlap',
           'overlap_icds'
        ]]
    return final


In [ ]:
#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).


def get_phecode_donor_file():
    # ---------- File 1 ----------
    wide1 = pd.read_csv(f"{data}/mcc2_phecode_table.csv", dtype={"person_id": str})

    # melt into long format
    donor_codes1 = wide1.melt(
        id_vars="person_id",
        var_name="phecode",
        value_name="has_code"
    )

    return donor_codes1


In [ ]:
#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).


def get_phecodeX_donor_file():
    
    wide2 = pd.read_csv(f"{data}/mcc2_phecodex_table.csv", dtype={"person_id": str}, usecols=lambda c: c != "sex")

    # melt into long format
    donor_codes2 = wide2.melt(
        id_vars="person_id",
        var_name="phecode",
        value_name="has_code"
    )

    return donor_codes2

In [ ]:
#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).
## donor_codes is the pandas version of "dict1: ICD -> list of donors"
### columns: donor_id, concept_code

def get_phecode_donor_ICD_yes_maps(donor_codes1, donor_codes2):

    # keep only codes that are present (True / 1)
    donor_codes1 = donor_codes1[donor_codes1["has_code"].astype(bool)].drop(columns="has_code")

    # keep only codes that are present (True / 1)
    donor_codes2 = donor_codes2[donor_codes2["has_code"].astype(bool)].drop(columns="has_code")
    
    
    #concat files
    donor_codes = pd.concat([donor_codes1, donor_codes2], ignore_index=True)

    #set column type as str
    donor_codes['phecode'] = donor_codes['phecode'].astype(str)
    donor_codes["person_id"] = donor_codes["person_id"].astype(str)
    donor_codes = donor_codes.drop_duplicates()

    return donor_codes


In [ ]:

# all B ICDs (phecode & phecodex), one row per phecode–ICD
# join B ICDs to donors
def get_phecode_donor_icd_phegroup_map(phecode_donor_map):

    phe_donors = phe_codes.merge(
        phecode_donor_map,
        on="concept_code",
        how="left"
    ).dropna(subset=["person_id"])
    
    

    # one row per (B, donor)
    phe_donors_unique = phe_donors[["phecode","concept_code", "person_id"]].drop_duplicates()

    # count donors per B (Phenotype B side, independent of A)
    b_counts = (
        phe_donors_unique
        .groupby(["concept_code", "phecode"])["person_id"]
        .nunique()
        .rename("n_phe_donors")
        .reset_index()
    )
    
    return phe_donors, b_counts


In [ ]:
 #get donor A count
    
def get_cohort_donor_ICD_map(filtered_cohort_dict):
  

    a_donors =  pd.concat(
    [
        df[["person_id"]].drop_duplicates().assign(
            condition_concept_id=cond_id,
            standard_concept_name=name,
        )
        for (cond_id, name), df in filtered_cohort_dict.items()
    ],
    ignore_index=True,
        )

    a_donors["person_id"] = a_donors["person_id"].astype(str)
    
    
    
    return a_donors

In [ ]:
a_counts = (
        cohort_donor_map
        .groupby(["condition_concept_id", "standard_concept_name"])["person_id"]
        .nunique()
        .rename("cohort_count")
        .reset_index()
    )

In [ ]:
a_counts

In [ ]:
icd_overlap_frac

In [ ]:
final_icd = get_cohort_phecode_ICD_overlap_frac_count(icd_overlap_count, cohort_icd_count)
# columns: condition_concept_id, standard_concept_name, phecode, n_my_icds, n_overlap, frac_overlap, overlap_icds
#final_icd

In [ ]:
valid_codes = icd_overlap_frac['phecode']
phecode_donor_map_filtered = phecode_donor_map[phecode_donor_map["phecode"].isin(valid_codes)]



In [ ]:
phecode_donor_map_filtered

In [ ]:
pairs = icd_overlap_frac[["condition_concept_id", "standard_concept_name", "phecode"]].drop_duplicates()
pairs["phecode"] = pairs["phecode"].astype(str)

# attach A donors to each pair
pairs_A = pairs.merge(cohort_donor_map, on=["condition_concept_id", "standard_concept_name"], how="left")

# attach B donors by phecode
pairs_AB = pairs_A.merge(phecode_donor_map_filtered, on=["phecode", "person_id"], how="inner")

# now count donors per (A,B)
cohort_phecode_sample_overlap_counts = (
    pairs_AB
    .groupby(["condition_concept_id", "standard_concept_name", "phecode"])["person_id"]
    .nunique()
    .rename("n_overlap_donors")
    .reset_index()
)


In [ ]:
#pairs
#pairs_A
#phecode_donor_map
#pairs_AB
cohort_phecode_sample_overlap_counts

In [ ]:
final_df = cohort_phecode_sample_overlap_counts.merge(icd_overlap_frac, on = ["condition_concept_id", "standard_concept_name", "phecode"])
                                                      

In [ ]:
final_df2 = final_df.merge(a_counts, on =["condition_concept_id", "standard_concept_name"])

In [ ]:
final_df2

In [ ]:
per_phecode_donor_count = phecode_donor_map_filtered.merge(phe_codes, on = "phecode")


In [ ]:
per_phecode_donor_count

In [ ]:
import pandas as pd

# 1) One row per (person_id, phecode)
phe_person = per_phecode_donor_count.drop_duplicates(subset=["person_id", "phecode"])

# 2) Count people per phecode (optionally keep description/group)
phe_counts = (
    phe_person
    .groupby(["phecode", "description", "group"], as_index=False)["person_id"]
    .nunique()
    .rename(columns={"person_id": "n_persons"})
)


In [ ]:
# one row per (B, donor)
phe_donors_unique = per_phecode_donor_count[["phecode","concept_code","description", "person_id"]].drop_duplicates()

    # count donors per B (Phenotype B side, independent of A)
b_counts = (
        phe_donors_unique
        .groupby(["phecode","description"])["person_id"]
        .nunique()
        .rename("n_phe_donors")
        .reset_index()
    )

In [ ]:
phe_counts

In [ ]:
final_final = final_df2.merge(phe_counts, on = "phecode", how = "left")

In [ ]:
final_final

In [ ]:
# fractions (avoid division by zero)
result["frac_A"] = result["n_overlap_donors"] / result["n_donors_A"].replace(0, pd.NA)
result["frac_B"] = result["n_overlap_donors"] / result["n_donors_B"].replace(0, pd.NA)

# keep the columns you said you want
out = result[[
    "condition_concept_id",   # Phenotype A
    "phecode",                # Phenotype B
    "n_overlap_donors",       # Overlap
    "frac_A",                 # Phenotype A fraction
    "frac_B",                 # Phenotype B fraction
]]


In [ ]:
final_final["Phecode_overlap_%"] = final_final

In [ ]:
b_counts["concept_code"] = b_counts["concept_code"].astype(str)

# Make all elements in overlap_icds lists into strings too
final_df2["overlap_icds"] = final_df2["overlap_icds"].apply(
    lambda codes: [str(c) for c in codes]
)


In [ ]:
overlap_long = final_df2.explode("overlap_icds").rename(
    columns={"overlap_icds": "concept_code"}
)
overlap_long

In [ ]:
final_merged = overlap_long.merge(
    b_counts,
    on=["concept_code", "phecode"],
    how="left",   # or "left" if you want to keep all overlap_long rows
)


In [ ]:
final_merged

In [ ]:
codes_count_per_phecode = (
    phe_codes
    .groupby(["phecode", "description"])["concept_code"]
    .nunique()
    .rename("ICD_count")
    .reset_index()
)

In [ ]:
codes_count_per_phecode

In [ ]:
group_cols = ["condition_concept_id", "standard_concept_name", "phecode"]

collapsed = (
    final_merged
    .groupby(group_cols, as_index=False)
    .agg(
        n_overlap_donors=("n_overlap_donors", "first"),
        n_my_icds=("n_my_icds", "first"),
        n_overlap=("n_overlap", "first"),
        frac_overlap=("frac_overlap", "first"),
        cohort_count=("cohort_count", "first"),
        description=("description", "first"),  # same within group in your example
        # collect all ICD codes into a list, deduplicated, order-preserving
        overlap_icds=("concept_code", lambda x: list(dict.fromkeys(x))),
        n_phe_donors=("n_phe_donors", "first"),
    )
)



In [ ]:
collasped

In [ ]:
answer = collapsed.merge(codes_count_per_phecode, on = ["phecode", "description"], how = "left")


In [ ]:
answer

In [ ]:
hep_c = answer[answer["standard_concept_name"] == "Viral hepatitis C"]

In [ ]:
hep_c

In [ ]:
#




# start from max_overlap so we keep exactly those (A,B) pairs from File 3
result = max_overlap[["condition_concept_id", "phecode"]].drop_duplicates()

# add counts and overlaps
result = result.merge(a_counts, on="condition_concept_id", how="left")
result = result.merge(b_counts, on=["condition_concept_id", "phecode"], how="left")
result = result.merge(overlap_counts_donors, on=["condition_concept_id", "phecode"], how="left")

# fill missing counts/overlaps with 0
for col in ["n_donors_A", "n_donors_B", "n_overlap_donors"]:
    result[col] = result[col].fillna(0).astype(int)

# fractions (avoid division by zero)
result["frac_A"] = result["n_overlap_donors"] / result["n_donors_A"].replace(0, pd.NA)
result["frac_B"] = result["n_overlap_donors"] / result["n_donors_B"].replace(0, pd.NA)

# keep the columns you said you want
out = result[[
    "condition_concept_id",   # Phenotype A
    "phecode",                # Phenotype B
    "n_overlap_donors",       # Overlap
    "frac_A",                 # Phenotype A fraction
    "frac_B",                 # Phenotype B fraction
]]


result = result.merge(
    max_overlap[["condition_concept_id", "phecode", "frac_overlap"]],
    on=["condition_concept_id", "phecode"],
    how="left"
)



In [ ]:
hep_cc = phe_codes[phe_codes["phecode"] == 070.3]

In [ ]:
hep_cc

In [ ]:
#function calls

'''
phe, phe_icd, phex, phex_icd = import_map_files()
phe_map = merge_phecodes_with_phenotype(phe, phe_icd)
phex_map = merge_phecodes_with_phenotype(phex, phex_icd)



cohort = get_data_pkl(data2, 'cohort_cond_df.pkl')
cohort_dict = get_cond_cohort_df_dict(cohort)
filtered_cohort_dict = filter_cond_df_dict(cohort_dict)

summary = get_icd_vocab_count_per_cohort(filtered_cohort_dict)

cohort_ICD_map = get_cohort_concept_id_to_ICD_mapping(cohort)

#get max overlap file: 

cond_codes, phe_codes = get_unique_phecodes_and_cohort_ICDs(cohort_ICD_map,phe_map, phex_map)


cohort_icd_count = get_cohort_ICD_codes_count_per_unique_condition(cond_codes)

icd_overlap = get_cohort_phecode_ICD_overlap(cond_codes, phe_codes)
icd_overlap_count = get_ICD_overlap_count(icd_overlap)
icd_overlap_frac = get_cohort_phecode_ICD_overlap_frac_count(icd_overlap_count, cohort_icd_count)


donor1 = get_phecode_donor_file()
donor2 = get_phecodeX_donor_file()


phecode_donor_map = get_phecode_donor_ICD_yes_maps(donor1, donor2)

cohort_donor_map = get_cohort_donor_ICD_map(filtered_cohort_dict)
'''

phe_donors, b_counts = get_phecode_donor_icd_phegroup_map(phecode_donor_map)


In [ ]:
##visualize


#phe_map
#phex_map
#cohort
#filtered_cohort_dict
#summary
#flat_phe_map
#cohort_ICD_map

#cond_codes
#phe_codes
#cohort_icd_count

#icd_overlap 
#icd_overlap_count 
icd_overlap_frac

#cohort_donor_map
#phecode_donor_map

In [ ]:
cohort_ICD_map

In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
import pandas
import os

# This query represents dataset "viral hep c phecode mapping" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_67312876_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1567375, 1567379, 35205771, 44819378, 44819379, 44823980, 44828695, 44832084, 44835626, 45547442, 45576259, 45581151, 45605221)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (45576259, 44835626, 1567379, 44832084, 45581151, 44823980, 1567375, 45605221, 44819379, 45547442, 35205771, 44819378, 44828695)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 0 
                                AND is_selectable = 1) 
                            AND is_standard = 0 )) criteria ) )
            )) c_occurrence 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_67312876_condition_df = pandas.read_gbq(
    dataset_67312876_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_67312876_condition_df.head(5)

In [ ]:
dataset_67312876_condition_df

In [ ]:
see = dataset_67312876_condition_df[dataset_67312876_condition_df["person_id"] == 1507121 ]



In [ ]:
see2 = cohort[cohort["person_id"] == 4340542 ]

In [ ]:
see2

In [ ]:
cohort

In [ ]:
hep_c = filtered_cohort_dict[(197494, 'Viral hepatitis C')]

In [ ]:
hep_c

In [ ]:

hep_ids = hep_c["person_id"].unique()

os_test = dataset_67312876_condition_df[
    ~dataset_67312876_condition_df["person_id"].isin(hep_ids)
]




In [ ]:


os_test2 = dataset_67312876_condition_df.merge(hep_c, how = "left")
   




In [ ]:
df1 = os_test[os_test["standard_concept_name"] == "Viral hepatitis C"]

In [ ]:
os_test["standard_concept_name"].unique()


In [ ]:
import pandas
import os

# This query represents dataset "fw" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_02547632_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1567375, 1567379, 35205771, 44819378, 44819379, 44823980, 44828695, 44832084, 44835626, 45547442, 45576259, 45581151, 45605221)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_array_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (440029)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_02547632_condition_df = pandas.read_gbq(
    dataset_02547632_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_02547632_condition_df.head(5)

In [ ]:
dataset_02547632_condition_df

In [ ]:
see3 = dataset_02547632_condition_df[dataset_02547632_condition_df["person_id"] == 1507121 ]



In [ ]:
see3